# 🛡️ ThreatShield AI
### Machine-learning network intrusion detection · KDD Cup 99

**How to run on Google Colab**
1. `Runtime → Run all` (or run each cell top to bottom).
2. Open the **public link** printed by the last cell (or use the inline app).
3. In the app: **Command Center** → load data → **Model Lab** → train → **Threat Analyzer** / **Traffic Scanner**.

The dataset downloads automatically (KDD Cup 99, 10%). You can also upload `kddcup.data_10_percent_corrected` inside the app.

## 1 · Install dependencies

In [ ]:
!pip install -q "gradio>=5.0,<6.0" scikit-learn pandas matplotlib numpy

## 2 · Core configuration & schema

In [ ]:
# ══════════════════════════════════════════════════════════════
#  ThreatShield AI  ·  Setup
# ══════════════════════════════════════════════════════════════
import warnings, time, io, os, glob, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

# ── Schema (identical to the original notebook) ────────────────
ALL_COLUMNS = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment",
    "urgent","hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted",
    "num_root","num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
    "dst_host_count","dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
]

ATTACK_MAP = {
    "normal":"normal","back":"dos","buffer_overflow":"u2r","ftp_write":"r2l","guess_passwd":"r2l",
    "imap":"r2l","ipsweep":"probe","land":"dos","loadmodule":"u2r","multihop":"r2l","neptune":"dos",
    "nmap":"probe","perl":"u2r","phf":"r2l","pod":"dos","portsweep":"probe","rootkit":"u2r",
    "satan":"probe","smurf":"dos","spy":"r2l","teardrop":"dos","warezclient":"r2l","warezmaster":"r2l",
}

PROTO_MAP = {"icmp": 0, "tcp": 1, "udp": 2}
FLAG_MAP  = {"SF":0,"S0":1,"REJ":2,"RSTR":3,"RSTO":4,"SH":5,"S1":6,"S2":7,"RSTOS0":8,"S3":9,"OTH":10}

# Columns removed in the original pipeline (correlation / constant / high-cardinality)
DROP_COLS = ["num_root","srv_serror_rate","srv_rerror_rate","dst_host_srv_serror_rate",
             "dst_host_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
             "dst_host_same_srv_rate","is_host_login","num_outbound_cmds","service"]
FEATURES = [c for c in ALL_COLUMNS if c not in DROP_COLS]          # 30 model features
CATEGORICAL = ["protocol_type", "flag"]

CLASS_INFO = {
    "normal": dict(name="Normal Traffic", sev="SAFE",     color="#22c55e",
                   desc="Traffic matches legitimate behaviour patterns. No action required."),
    "dos":    dict(name="Denial of Service (DoS)", sev="HIGH", color="#f97316",
                   desc="Flooding behaviour intended to exhaust bandwidth, memory or connection tables (e.g. neptune, smurf, back, teardrop)."),
    "probe":  dict(name="Probing / Reconnaissance", sev="MEDIUM", color="#eab308",
                   desc="Network or port scanning to map hosts and discover vulnerable services (e.g. nmap, portsweep, ipsweep, satan)."),
    "r2l":    dict(name="Remote-to-Local (R2L)", sev="HIGH", color="#ef4444",
                   desc="Unauthorised remote access attempts such as password guessing or FTP abuse (e.g. guess_passwd, warezclient, ftp_write)."),
    "u2r":    dict(name="User-to-Root (U2R)", sev="CRITICAL", color="#dc2626",
                   desc="Privilege-escalation attempts from a normal account to root (e.g. buffer_overflow, rootkit, loadmodule)."),
}

MODEL_FACTORY = {
    "Naive Bayes":         lambda: GaussianNB(),
    "Decision Tree":       lambda: DecisionTreeClassifier(criterion="entropy", max_depth=4),
    "Random Forest":       lambda: RandomForestClassifier(n_estimators=30, n_jobs=-1, random_state=0),
    "Logistic Regression": lambda: LogisticRegression(max_iter=2000),
    "Gradient Boosting":   lambda: GradientBoostingClassifier(random_state=0),
    "SVM":                 lambda: SVC(gamma="scale"),
}
SLOW_MODELS = {"SVM", "Gradient Boosting"}
CLASSES = ["normal", "dos", "probe", "r2l", "u2r"]

# Global application state
S = dict(df=None, raw_train=None, raw_test=None, y_train=None, y_test=None,
         X_train=None, X_test=None, scaler=None, models={}, results={}, source="—")

print(f"✅ ThreatShield AI core ready · gradio {gr.__version__} · {len(FEATURES)} model features")


## 3 · Data loading & preprocessing

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Data loading & preprocessing (mirrors the original notebook)
# ══════════════════════════════════════════════════════════════
NUMERIC_FEATURES = [c for c in FEATURES if c not in CATEGORICAL]

def _clean_frame(df):
    """Coerce dtypes, derive 'Attack Type', drop unknown labels."""
    for c in ["protocol_type", "service", "flag", "target"]:
        df[c] = df[c].apply(lambda v: v.decode() if isinstance(v, bytes) else str(v)).str.strip()
    df["target"] = df["target"].str.rstrip(".")
    for c in ALL_COLUMNS:
        if c not in ("protocol_type", "service", "flag"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["Attack Type"] = df["target"].map(ATTACK_MAP)
    df = df.dropna(subset=["Attack Type"]).reset_index(drop=True)
    return df

def read_kdd_file(path):
    df = pd.read_csv(path, names=ALL_COLUMNS + ["target"], header=None)
    return _clean_frame(df)

def download_kdd():
    from sklearn.datasets import fetch_kddcup99
    bunch = fetch_kddcup99(percent10=True, as_frame=False, download_if_missing=True)
    df = pd.DataFrame(bunch.data, columns=ALL_COLUMNS)
    df["target"] = bunch.target
    return _clean_frame(df)

def transform_raw(raw):
    """raw feature frame (strings for protocol/flag) -> scaled numpy matrix."""
    X = raw[FEATURES].copy()
    X["protocol_type"] = X["protocol_type"].astype(str).str.lower().map(PROTO_MAP)
    X["flag"] = X["flag"].astype(str).str.upper().map(FLAG_MAP)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    return S["scaler"].transform(X.values)

def prepare(df, source):
    raw = df[FEATURES]
    y = df["Attack Type"]
    raw_tr, raw_te, y_tr, y_te = train_test_split(raw, y, test_size=0.33, random_state=42)
    X = raw_tr.copy()
    X["protocol_type"] = X["protocol_type"].map(PROTO_MAP)
    X["flag"] = X["flag"].map(FLAG_MAP)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    S["scaler"] = MinMaxScaler().fit(X.values)
    S.update(df=df, raw_train=raw_tr, raw_test=raw_te, y_train=y_tr.reset_index(drop=True),
             y_test=y_te.reset_index(drop=True), source=source, models={}, results={})
    S["X_train"] = S["scaler"].transform(X.values)
    S["X_test"]  = transform_raw(raw_te)
    S["raw_test"] = raw_te.reset_index(drop=True)


## 4 · Visual theme & charts

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Visual theme + chart helpers
# ══════════════════════════════════════════════════════════════
BG, PANEL, GRID, TXT, MUTED = "#0b1220", "#111a2e", "#1e2a44", "#e2e8f0", "#94a3b8"
ACCENT, ACCENT2 = "#22d3ee", "#6366f1"
CLASS_COLORS = {"normal":"#22c55e","dos":"#f97316","probe":"#eab308","r2l":"#ef4444","u2r":"#dc2626"}
PALETTE = ["#22d3ee","#6366f1","#a78bfa","#f472b6","#34d399","#fbbf24"]

def _style(ax, title=None):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values(): s.set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.xaxis.label.set_color(MUTED); ax.yaxis.label.set_color(MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.6, alpha=0.7); ax.set_axisbelow(True)
    if title: ax.set_title(title, color=TXT, fontsize=12, fontweight="bold", loc="left", pad=12)

def _fig(w=6.2, h=3.6):
    fig, ax = plt.subplots(figsize=(w, h), facecolor=BG)
    return fig, ax

def _empty(msg):
    fig, ax = _fig()
    ax.set_facecolor(BG); ax.axis("off")
    ax.text(0.5, 0.5, msg, ha="center", va="center", color=MUTED, fontsize=11, transform=ax.transAxes)
    return fig

def plot_attack_dist(df):
    vc = df["Attack Type"].value_counts()
    fig, ax = _fig()
    bars = ax.bar(vc.index.str.upper(), vc.values, color=[CLASS_COLORS.get(i, ACCENT) for i in vc.index], width=0.6)
    ax.set_yscale("log"); _style(ax, "Attack-Class Distribution (log scale)")
    for b, v in zip(bars, vc.values):
        ax.text(b.get_x()+b.get_width()/2, v*1.15, f"{v:,}", ha="center", color=TXT, fontsize=9)
    fig.tight_layout(); return fig

def plot_protocol(df):
    ct = pd.crosstab(df["protocol_type"], df["Attack Type"])
    fig, ax = _fig()
    bottom = np.zeros(len(ct))
    for cls in [c for c in CLASSES if c in ct.columns]:
        ax.bar(ct.index.str.upper(), ct[cls].values, bottom=bottom, color=CLASS_COLORS[cls], label=cls.upper(), width=0.55)
        bottom += ct[cls].values
    _style(ax, "Protocol vs. Threat Class")
    leg = ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TXT, fontsize=8)
    fig.tight_layout(); return fig

def plot_top_labels(df):
    vc = df["target"].value_counts().head(10)[::-1]
    fig, ax = _fig()
    ax.barh(vc.index, vc.values, color=ACCENT2, height=0.6)
    ax.set_xscale("log"); _style(ax, "Top 10 Traffic Signatures")
    ax.grid(axis="x", color=GRID, linewidth=0.6); ax.grid(axis="y", visible=False)
    fig.tight_layout(); return fig

def plot_flags(df):
    vc = df["flag"].value_counts()
    fig, ax = _fig()
    ax.bar(vc.index, vc.values, color=ACCENT, width=0.6)
    ax.set_yscale("log"); _style(ax, "Connection Flag Distribution")
    fig.tight_layout(); return fig

def plot_model_compare(res):
    if not res: return _empty("Train models to compare accuracy")
    names = list(res); acc = [res[n]["test_acc"] for n in names]
    fig, ax = _fig(6.4, 3.8)
    bars = ax.bar(names, acc, color=PALETTE[:len(names)], width=0.6)
    lo = max(0, min(acc) - 5); ax.set_ylim(lo, 100.8)
    _style(ax, "Test Accuracy by Model (%)")
    for b, v in zip(bars, acc):
        ax.text(b.get_x()+b.get_width()/2, v+0.15, f"{v:.2f}", ha="center", color=TXT, fontsize=9, fontweight="bold")
    plt.setp(ax.get_xticklabels(), rotation=18, ha="right")
    fig.tight_layout(); return fig

def plot_time_compare(res):
    if not res: return _empty("Train models to compare speed")
    names = list(res)
    fig, ax = _fig(6.4, 3.8)
    x = np.arange(len(names)); w = 0.38
    ax.bar(x-w/2, [res[n]["train_time"] for n in names], w, color=ACCENT, label="Training")
    ax.bar(x+w/2, [res[n]["pred_time"] for n in names], w, color="#f472b6", label="Inference")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=18, ha="right")
    ax.set_yscale("log"); _style(ax, "Compute Time (seconds, log scale)")
    ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TXT, fontsize=8)
    fig.tight_layout(); return fig

def plot_confusion(name):
    r = S["results"].get(name)
    if r is None: return _empty("Train a model to view its confusion matrix")
    cm = r["cm"]; cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = _fig(5.6, 4.4)
    ax.imshow(cmn, cmap=matplotlib.colors.LinearSegmentedColormap.from_list("ts", [PANEL, "#155e75", ACCENT]), vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels([c.upper() for c in CLASSES]); ax.set_yticklabels([c.upper() for c in CLASSES])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); _style(ax, f"Confusion Matrix · {name}")
    ax.grid(False)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", fontsize=9,
                    color="white" if cmn[i, j] < 0.75 else "#0b1220", fontweight="bold")
    fig.tight_layout(); return fig

def plot_importance(name):
    m = S["models"].get(name)
    if m is None or not hasattr(m, "feature_importances_"):
        return _empty("Feature importance is available for tree-based models\n(Decision Tree · Random Forest · Gradient Boosting)")
    imp = pd.Series(m.feature_importances_, index=FEATURES).sort_values().tail(10)
    fig, ax = _fig(5.6, 4.4)
    ax.barh(imp.index, imp.values, color=ACCENT, height=0.6)
    _style(ax, f"Top Features · {name}")
    ax.grid(axis="x", color=GRID, linewidth=0.6); ax.grid(axis="y", visible=False)
    fig.tight_layout(); return fig

def plot_proba(classes, probs):
    order = np.argsort(probs)
    fig, ax = _fig(5.2, 3.0)
    ax.barh([str(classes[i]).upper() for i in order], [probs[i]*100 for i in order],
            color=[CLASS_COLORS.get(classes[i], ACCENT) for i in order], height=0.55)
    ax.set_xlim(0, 100); _style(ax, "Class Probability (%)")
    ax.grid(axis="x", color=GRID, linewidth=0.6); ax.grid(axis="y", visible=False)
    fig.tight_layout(); return fig

def plot_scan_summary(counts):
    fig, ax = _fig(5.2, 3.4)
    keys = [c for c in CLASSES if counts.get(c, 0) > 0]
    ax.bar([k.upper() for k in keys], [counts[k] for k in keys], color=[CLASS_COLORS[k] for k in keys], width=0.55)
    _style(ax, "Detected Traffic Composition")
    fig.tight_layout(); return fig


## 5 · Training & inference engine

In [ ]:
# ══════════════════════════════════════════════════════════════
#  Training + inference engine
# ══════════════════════════════════════════════════════════════
SLOW_TRAIN_CAP, SLOW_TEST_CAP = 40_000, 30_000

def train_models(selected, fast_mode, progress=gr.Progress()):
    if S["X_train"] is None:
        raise gr.Error("Load a dataset first (Command Center tab).")
    if not selected:
        raise gr.Error("Select at least one model.")
    rng = np.random.RandomState(0)
    Xtr, ytr, Xte, yte = S["X_train"], S["y_train"].values, S["X_test"], S["y_test"].values
    for i, name in enumerate(selected):
        progress(i / len(selected), desc=f"Training {name}…")
        xt, yt, xe, ye = Xtr, ytr, Xte, yte
        capped = False
        if fast_mode and name in SLOW_MODELS:
            ti = rng.choice(len(Xtr), min(SLOW_TRAIN_CAP, len(Xtr)), replace=False)
            ei = rng.choice(len(Xte), min(SLOW_TEST_CAP, len(Xte)), replace=False)
            xt, yt, xe, ye = Xtr[ti], ytr[ti], Xte[ei], yte[ei]; capped = True
        model = MODEL_FACTORY[name]()
        t0 = time.time(); model.fit(xt, yt); t_train = time.time() - t0
        t0 = time.time(); p_tr = model.predict(xt); p_te = model.predict(xe); t_pred = time.time() - t0
        S["models"][name] = model
        S["results"][name] = dict(
            train_acc=accuracy_score(yt, p_tr) * 100, test_acc=accuracy_score(ye, p_te) * 100,
            precision=precision_score(ye, p_te, average="macro", zero_division=0) * 100,
            recall=recall_score(ye, p_te, average="macro", zero_division=0) * 100,
            f1=f1_score(ye, p_te, average="macro", zero_division=0) * 100,
            train_time=t_train, pred_time=t_pred, capped=capped,
            cm=confusion_matrix(ye, p_te, labels=CLASSES))
    progress(1.0, desc="Done")
    trained = list(S["results"])
    best = max(S["results"], key=lambda n: S["results"][n]["test_acc"])
    return (results_table(), plot_model_compare(S["results"]), plot_time_compare(S["results"]),
            gr.update(choices=trained, value=best), plot_confusion(best), plot_importance(best),
            gr.update(choices=trained, value=best), gr.update(choices=trained, value=best),
            model_banner(best), status_html())

def results_table():
    rows = []
    for n, r in S["results"].items():
        rows.append([n + (" *" if r["capped"] else ""), round(r["train_acc"], 2), round(r["test_acc"], 2),
                     round(r["precision"], 2), round(r["recall"], 2), round(r["f1"], 2),
                     round(r["train_time"], 2), round(r["pred_time"], 2)])
    cols = ["Model", "Train Acc %", "Test Acc %", "Precision %", "Recall %", "F1 %", "Train s", "Inference s"]
    return pd.DataFrame(rows, columns=cols).sort_values("Test Acc %", ascending=False)

def show_model_details(name):
    return plot_confusion(name), plot_importance(name)

def model_banner(best):
    r = S["results"][best]
    return (f"<div class='ts-note'>🏆 <b>Best performer:</b> {best} — {r['test_acc']:.2f}% test accuracy, "
            f"{r['f1']:.2f}% macro-F1. &nbsp;<span style='color:#94a3b8'>* trained on a capped subset (Fast Mode).</span></div>")

# ── Single-record inference ───────────────────────────────────
def _verdict_html(label, conf, model_name):
    info = CLASS_INFO[label]; safe = label == "normal"
    icon = "🛡️" if safe else "🚨"
    title = "NO THREAT DETECTED" if safe else "THREAT DETECTED"
    conf_txt = f"{conf*100:.1f}%" if conf is not None else "n/a"
    return f"""
    <div class="ts-verdict" style="border-color:{info['color']};box-shadow:0 0 28px {info['color']}33">
      <div class="ts-v-head"><span class="ts-v-icon">{icon}</span>
        <div><div class="ts-v-title" style="color:{info['color']}">{title}</div>
        <div class="ts-v-sub">{info['name']}</div></div></div>
      <div class="ts-v-grid">
        <div><span>SEVERITY</span><b style="color:{info['color']}">{info['sev']}</b></div>
        <div><span>CONFIDENCE</span><b>{conf_txt}</b></div>
        <div><span>ENGINE</span><b>{model_name}</b></div>
      </div>
      <p class="ts-v-desc">{info['desc']}</p>
    </div>"""

def analyze_record(model_name, *vals):
    if model_name not in S["models"]:
        raise gr.Error("Train at least one model first (Model Lab tab).")
    row = pd.DataFrame([dict(zip(FEATURES, vals))])
    X = transform_raw(row)
    m = S["models"][model_name]
    label = str(m.predict(X)[0])
    if hasattr(m, "predict_proba"):
        pr = m.predict_proba(X)[0]; cls = list(m.classes_)
        conf = float(pr[cls.index(label)])
    else:   # SVM without probability → softmax over decision scores
        d = m.decision_function(X)[0]; e = np.exp(d - d.max()); pr = e / e.sum(); cls = list(m.classes_)
        conf = None
    return _verdict_html(label, conf, model_name), plot_proba(cls, pr)

def load_sample(kind):
    if S["raw_test"] is None:
        raise gr.Error("Load a dataset first (Command Center tab).")
    idx = np.where(S["y_test"].values == kind)[0]
    if len(idx) == 0:
        raise gr.Error(f"No '{kind}' samples in the test split.")
    row = S["raw_test"].iloc[int(random.choice(idx))]
    out = []
    for f in FEATURES:
        v = row[f]
        out.append(str(v) if f in CATEGORICAL else float(v))
    return out

# ── Batch / traffic scanning ──────────────────────────────────
def _scan(raw, truth, model_name):
    m = S["models"][model_name]
    preds = m.predict(transform_raw(raw))
    counts = pd.Series(preds).value_counts().to_dict()
    n = len(preds); threats = int((preds != "normal").sum())
    acc_html = ""
    if truth is not None:
        acc = (preds == truth).mean() * 100
        acc_html = f"<div class='ts-kpi'><span>ACCURACY vs. GROUND TRUTH</span><b>{acc:.2f}%</b></div>"
    kpi = f"""<div class="ts-kpis">
      <div class='ts-kpi'><span>RECORDS SCANNED</span><b>{n:,}</b></div>
      <div class='ts-kpi'><span>THREATS FLAGGED</span><b style='color:#f87171'>{threats:,}</b></div>
      <div class='ts-kpi'><span>THREAT RATE</span><b>{threats/max(n,1)*100:.1f}%</b></div>{acc_html}</div>"""
    out = raw.copy().reset_index(drop=True)
    out.insert(0, "verdict", [CLASS_INFO[p]["sev"] for p in preds])
    out.insert(0, "prediction", preds)
    if truth is not None: out.insert(1, "actual", truth)
    path = "/tmp/threatshield_scan_report.csv"
    out.to_csv(path, index=False)
    return kpi, plot_scan_summary(counts), out.head(500), path

def simulate_traffic(model_name, n):
    if model_name not in S["models"]:
        raise gr.Error("Train at least one model first (Model Lab tab).")
    n = int(min(n, len(S["raw_test"])))
    idx = np.random.RandomState().choice(len(S["raw_test"]), n, replace=False)
    return _scan(S["raw_test"].iloc[idx], S["y_test"].values[idx], model_name)

def scan_file(model_name, file):
    if model_name not in S["models"]:
        raise gr.Error("Train at least one model first (Model Lab tab).")
    if file is None:
        raise gr.Error("Upload a CSV of network connection records.")
    path = file if isinstance(file, str) else file.name
    df = pd.read_csv(path, header=None, low_memory=False)
    first = str(df.iloc[0, 0]).strip()
    if not first.replace(".", "", 1).isdigit():          # has a header row
        df = pd.read_csv(path, low_memory=False)
        if set(FEATURES).issubset(df.columns):
            raw = df[FEATURES]; truth = None
            for c in ("Attack Type", "target"):
                if c in df.columns:
                    t = df[c].astype(str).str.rstrip(".")
                    truth = (t.map(ATTACK_MAP).fillna(t) if c == "target" else t).values; break
            return _scan(raw, truth, model_name)
        df = pd.read_csv(path, header=None, low_memory=False)
    if df.shape[1] not in (41, 42):
        raise gr.Error(f"Expected 41 KDD features (+ optional label) — found {df.shape[1]} columns.")
    truth = None
    if df.shape[1] == 42:
        t = df.iloc[:, 41].astype(str).str.rstrip(".")
        truth = t.map(ATTACK_MAP).fillna(t).values
    df = df.iloc[:, :41]; df.columns = ALL_COLUMNS
    return _scan(df[FEATURES], truth, model_name)

# ── Dataset loading ───────────────────────────────────────────
def kpi_cards(df):
    n = len(df); atk = int((df["Attack Type"] != "normal").sum())
    return f"""<div class="ts-kpis">
      <div class='ts-kpi'><span>TOTAL RECORDS</span><b>{n:,}</b></div>
      <div class='ts-kpi'><span>ATTACK RECORDS</span><b style='color:#f87171'>{atk:,}</b></div>
      <div class='ts-kpi'><span>NORMAL RECORDS</span><b style='color:#4ade80'>{n-atk:,}</b></div>
      <div class='ts-kpi'><span>SIGNATURES</span><b>{df['target'].nunique()}</b></div>
      <div class='ts-kpi'><span>MODEL FEATURES</span><b>{len(FEATURES)}</b></div></div>"""

def load_data(source, file, progress=gr.Progress()):
    progress(0.1, desc="Loading dataset…")
    if source.startswith("Upload"):
        if file is None:
            path = next(iter(glob.glob("/content/kddcup.data_10_percent*") + glob.glob("kddcup.data_10_percent*")), None)
            if path is None: raise gr.Error("Upload the kddcup.data_10_percent_corrected file, or choose the auto-download option.")
        else:
            path = file if isinstance(file, str) else file.name
        df = read_kdd_file(path); src = os.path.basename(path)
    else:
        try:
            df = download_kdd(); src = "KDD Cup 99 · 10% (sklearn)"
        except Exception as e:
            raise gr.Error(f"Download failed ({e}). Upload the dataset file instead.")
    progress(0.6, desc="Preprocessing & scaling…")
    prepare(df, src)
    progress(1.0)
    return (kpi_cards(df), plot_attack_dist(df), plot_protocol(df), plot_top_labels(df), plot_flags(df),
            df.head(12).drop(columns=["target"]), status_html())

def status_html():
    ds = "ok" if S["df"] is not None else "off"
    md = "ok" if S["models"] else "off"
    return f"""<div class="ts-status">
      <span class="pill {ds}">● DATASET {'LOADED · ' + S['source'] if ds=='ok' else 'NOT LOADED'}</span>
      <span class="pill {md}">● {len(S['models'])} MODEL(S) ARMED</span>
      <span class="pill ok">● ENGINE ONLINE</span></div>"""


## 6 · User interface

In [ ]:
# ══════════════════════════════════════════════════════════════
#  ThreatShield AI · User Interface
# ══════════════════════════════════════════════════════════════
CSS = """
:root { --ts-bg:#0b1220; --ts-panel:#111a2e; --ts-line:#1e2a44; --ts-txt:#e2e8f0; --ts-mut:#94a3b8; --ts-acc:#22d3ee; --ts-acc2:#6366f1; }
.gradio-container { max-width: 1280px !important; margin: auto; background: var(--ts-bg) !important; font-family: 'Inter', system-ui, sans-serif !important; }
footer { display: none !important; }

.ts-hero { position:relative; overflow:hidden; display:flex; align-items:center; gap:22px; padding:28px 34px; border-radius:18px;
  background: radial-gradient(1200px 300px at 0% 0%, rgba(34,211,238,.18), transparent 60%),
              radial-gradient(900px 300px at 100% 100%, rgba(99,102,241,.22), transparent 60%), linear-gradient(135deg,#0f1a33,#0b1220);
  border:1px solid var(--ts-line); margin-bottom:12px; }
.ts-hero::after { content:""; position:absolute; inset:0; background-image:linear-gradient(rgba(148,163,184,.05) 1px,transparent 1px),linear-gradient(90deg,rgba(148,163,184,.05) 1px,transparent 1px); background-size:34px 34px; pointer-events:none; }
.ts-logo { width:64px; height:64px; flex:none; filter: drop-shadow(0 0 14px rgba(34,211,238,.55)); }
.ts-brand { font-size:34px; font-weight:800; letter-spacing:-.5px; color:#fff; line-height:1.1; }
.ts-brand span { background:linear-gradient(90deg,#22d3ee,#818cf8); -webkit-background-clip:text; background-clip:text; color:transparent; }
.ts-tag { color:var(--ts-mut); font-size:14px; margin-top:6px; letter-spacing:.3px; }
.ts-badges { margin-left:auto; display:flex; gap:8px; z-index:1; flex-wrap:wrap; justify-content:flex-end; }
.ts-badge { font-size:11px; font-weight:600; letter-spacing:.8px; padding:6px 12px; border-radius:999px; color:#a5f3fc; border:1px solid rgba(34,211,238,.35); background:rgba(34,211,238,.08); }

.ts-status { display:flex; gap:10px; flex-wrap:wrap; margin:0 0 10px 0; }
.pill { font-size:11px; font-weight:700; letter-spacing:.8px; padding:6px 12px; border-radius:999px; border:1px solid var(--ts-line); background:var(--ts-panel); color:var(--ts-mut); }
.pill.ok { color:#4ade80; border-color:rgba(74,222,128,.35); background:rgba(74,222,128,.07); }
.pill.off { color:#fbbf24; border-color:rgba(251,191,36,.35); background:rgba(251,191,36,.07); }

.ts-kpis { display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:12px; margin:6px 0 4px; }
.ts-kpi { background:linear-gradient(180deg,#131f38,#0f1830); border:1px solid var(--ts-line); border-radius:14px; padding:16px 18px; }
.ts-kpi span { display:block; font-size:10.5px; letter-spacing:1.1px; color:var(--ts-mut); font-weight:600; }
.ts-kpi b { display:block; font-size:28px; margin-top:6px; color:#fff; font-weight:800; }

.ts-note { background:rgba(34,211,238,.07); border:1px solid rgba(34,211,238,.25); color:var(--ts-txt); padding:12px 16px; border-radius:12px; font-size:14px; }
.ts-h { color:#fff; font-size:18px; font-weight:700; margin:4px 0 2px; }
.ts-sub { color:var(--ts-mut); font-size:13.5px; margin-bottom:6px; }

.ts-verdict { background:linear-gradient(180deg,#131f38,#0f1830); border:1.5px solid; border-radius:16px; padding:22px; }
.ts-v-head { display:flex; gap:16px; align-items:center; }
.ts-v-icon { font-size:44px; }
.ts-v-title { font-size:22px; font-weight:800; letter-spacing:1px; }
.ts-v-sub { color:var(--ts-txt); font-size:15px; margin-top:2px; }
.ts-v-grid { display:grid; grid-template-columns:repeat(3,1fr); gap:10px; margin:18px 0 12px; }
.ts-v-grid div { background:rgba(255,255,255,.03); border:1px solid var(--ts-line); border-radius:10px; padding:10px 12px; }
.ts-v-grid span { display:block; font-size:10px; letter-spacing:1px; color:var(--ts-mut); font-weight:600; }
.ts-v-grid b { font-size:16px; color:#fff; }
.ts-v-desc { color:var(--ts-mut); font-size:13.5px; line-height:1.55; margin:0; }

.ts-glossary { display:grid; grid-template-columns:repeat(auto-fit,minmax(240px,1fr)); gap:12px; }
.ts-card { background:var(--ts-panel); border:1px solid var(--ts-line); border-left:4px solid; border-radius:12px; padding:14px 16px; color:var(--ts-txt); font-size:13.5px; line-height:1.5; }
.ts-card b { display:block; font-size:15px; margin-bottom:4px; }
.ts-steps { color:var(--ts-txt); font-size:14px; line-height:1.9; }
.ts-steps code { background:#0b1220; border:1px solid var(--ts-line); padding:1px 7px; border-radius:6px; color:#67e8f9; }

button.primary, .primary { background:linear-gradient(90deg,#06b6d4,#6366f1) !important; border:none !important; color:#fff !important; font-weight:700 !important; letter-spacing:.4px; }
button.primary:hover { filter:brightness(1.12); transform:translateY(-1px); }
.tab-nav button, button[role="tab"] { font-weight:600 !important; letter-spacing:.3px; }
.ts-foot { text-align:center; color:#64748b; font-size:12px; padding:18px 0 6px; }
"""

LOGO = """<svg class="ts-logo" viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">
<defs><linearGradient id="g" x1="0" y1="0" x2="1" y2="1"><stop offset="0" stop-color="#22d3ee"/><stop offset="1" stop-color="#6366f1"/></linearGradient></defs>
<path d="M32 4 8 13v17c0 15 10 26 24 30 14-4 24-15 24-30V13L32 4z" fill="rgba(34,211,238,.08)" stroke="url(#g)" stroke-width="3" stroke-linejoin="round"/>
<path d="M18 33h8l4-9 6 17 4-8h6" fill="none" stroke="url(#g)" stroke-width="3.2" stroke-linecap="round" stroke-linejoin="round"/></svg>"""

HEADER = f"""<div class="ts-hero">{LOGO}
  <div><div class="ts-brand">Threat<span>Shield</span> AI</div>
  <div class="ts-tag">Machine-learning powered network intrusion detection · KDD Cup 99</div></div>
  <div class="ts-badges"><span class="ts-badge">6 ML ENGINES</span><span class="ts-badge">5 THREAT CLASSES</span><span class="ts-badge">REAL-TIME ANALYSIS</span></div>
</div>"""

DEFAULTS = dict(protocol_type="tcp", flag="SF", duration=0, src_bytes=181, dst_bytes=5450, logged_in=1, count=8,
                srv_count=8, same_srv_rate=1.0, dst_host_count=9, dst_host_srv_count=9, dst_host_same_src_port_rate=0.11)
ESSENTIAL = ["protocol_type","flag","duration","src_bytes","dst_bytes","logged_in","count","srv_count",
             "serror_rate","rerror_rate","same_srv_rate","dst_host_count","dst_host_srv_count"]

theme = gr.themes.Base(primary_hue=gr.themes.colors.cyan, neutral_hue=gr.themes.colors.slate,
                       font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"]).set(
    body_background_fill_dark="#0b1220", background_fill_primary_dark="#111a2e", background_fill_secondary_dark="#0b1220",
    block_background_fill_dark="#111a2e", block_border_color_dark="#1e2a44", border_color_primary_dark="#1e2a44",
    input_background_fill_dark="#0b1220", block_label_background_fill_dark="#111a2e")

with gr.Blocks(theme=theme, css=CSS, title="ThreatShield AI",
               js="() => { document.body.classList.add('dark'); }") as demo:

    gr.HTML(HEADER)
    status = gr.HTML(status_html())

    with gr.Tabs():
        # ───────────── 1 · COMMAND CENTER ─────────────
        with gr.Tab("📊  Command Center"):
            gr.HTML("<div class='ts-h'>Data Ingestion</div><div class='ts-sub'>Load the KDD Cup 99 (10%) dataset — about 494K labelled network connections. Auto-download works out of the box in Colab.</div>")
            with gr.Row():
                src = gr.Radio(["Auto-download KDD Cup 99 (10%)", "Upload dataset file"], value="Auto-download KDD Cup 99 (10%)", label="Data source", scale=2)
                up = gr.File(label="kddcup.data_10_percent_corrected (optional)", scale=2)
            load_btn = gr.Button("⚡  Load & Preprocess Dataset", variant="primary", size="lg")
            kpis = gr.HTML()
            with gr.Row():
                p1 = gr.Plot(label="Threat classes"); p2 = gr.Plot(label="Protocols")
            with gr.Row():
                p3 = gr.Plot(label="Signatures"); p4 = gr.Plot(label="Flags")
            preview = gr.Dataframe(label="Data preview", interactive=False, wrap=False)

        # ───────────── 2 · MODEL LAB ─────────────
        with gr.Tab("🧠  Model Lab"):
            gr.HTML("<div class='ts-h'>Detection Engines</div><div class='ts-sub'>Train and benchmark the same six classifiers from the original notebook on a 67 / 33 split with MinMax scaling.</div>")
            with gr.Row():
                model_sel = gr.CheckboxGroup(list(MODEL_FACTORY), value=["Naive Bayes","Decision Tree","Random Forest","Logistic Regression"],
                                             label="Select engines", scale=3)
                with gr.Column(scale=2):
                    fast = gr.Checkbox(True, label="Fast mode for SVM & Gradient Boosting",
                                       info=f"Trains on {SLOW_TRAIN_CAP:,} rows and tests on {SLOW_TEST_CAP:,} (full run takes ~5+ minutes each).")
                    train_btn = gr.Button("🚀  Train & Evaluate", variant="primary", size="lg")
            banner = gr.HTML()
            table = gr.Dataframe(label="Leaderboard", interactive=False, wrap=True)
            with gr.Row():
                acc_plot = gr.Plot(label="Accuracy"); time_plot = gr.Plot(label="Compute time")
            detail_model = gr.Dropdown([], label="Inspect model")
            with gr.Row():
                cm_plot = gr.Plot(label="Confusion matrix"); imp_plot = gr.Plot(label="Feature importance")

        # ───────────── 3 · THREAT ANALYZER ─────────────
        with gr.Tab("🔍  Threat Analyzer"):
            gr.HTML("<div class='ts-h'>Single-Connection Analysis</div><div class='ts-sub'>Inspect one network connection. Load a real sample from the test set or edit the features manually.</div>")
            with gr.Row():
                an_model = gr.Dropdown([], label="Detection engine", scale=2)
                sample_kind = gr.Dropdown(CLASSES, value="dos", label="Sample class", scale=1)
                sample_btn = gr.Button("🎲  Load random sample", scale=1)
            comps = {}
            def make(f):
                if f == "protocol_type": return gr.Dropdown(["tcp","udp","icmp"], value="tcp", label=f)
                if f == "flag": return gr.Dropdown(list(FLAG_MAP), value="SF", label=f)
                return gr.Number(value=DEFAULTS.get(f, 0), label=f)
            with gr.Accordion("Connection essentials", open=True):
                for i in range(0, len(ESSENTIAL), 4):
                    with gr.Row():
                        for f in ESSENTIAL[i:i+4]: comps[f] = make(f)
            rest = [f for f in FEATURES if f not in ESSENTIAL]
            with gr.Accordion("Advanced features", open=False):
                for i in range(0, len(rest), 4):
                    with gr.Row():
                        for f in rest[i:i+4]: comps[f] = make(f)
            analyze_btn = gr.Button("🛡️  Analyze Connection", variant="primary", size="lg")
            with gr.Row():
                verdict = gr.HTML(); proba = gr.Plot(label="Class probabilities")
            ordered = [comps[f] for f in FEATURES]

        # ───────────── 4 · TRAFFIC SCANNER ─────────────
        with gr.Tab("📡  Traffic Scanner"):
            gr.HTML("<div class='ts-h'>Bulk Traffic Scan</div><div class='ts-sub'>Scan a CSV of KDD-format connection records, or simulate live traffic from the held-out test set.</div>")
            with gr.Row():
                sc_model = gr.Dropdown([], label="Detection engine", scale=2)
                n_sim = gr.Slider(100, 20000, value=2000, step=100, label="Simulated records", scale=2)
                sim_btn = gr.Button("📡  Simulate live traffic", variant="primary", scale=1)
            with gr.Row():
                csv_in = gr.File(label="Upload CSV (41 KDD features, optional label column, header optional)", scale=3)
                scan_btn = gr.Button("🔎  Scan file", variant="primary", scale=1)
            scan_kpi = gr.HTML()
            with gr.Row():
                scan_plot = gr.Plot(label="Composition"); scan_file_out = gr.File(label="Download scan report")
            scan_table = gr.Dataframe(label="Scan results (first 500 rows)", interactive=False)

        # ───────────── 5 · KNOWLEDGE BASE ─────────────
        with gr.Tab("📘  Knowledge Base"):
            cards = "".join(f"<div class='ts-card' style='border-left-color:{v['color']}'><b style='color:{v['color']}'>{v['name']} · {v['sev']}</b>{v['desc']}</div>"
                            for v in CLASS_INFO.values())
            gr.HTML(f"""<div class='ts-h'>Threat Taxonomy</div><div class='ts-sub'>The five traffic classes ThreatShield AI distinguishes.</div>
              <div class='ts-glossary'>{cards}</div>
              <div class='ts-h' style='margin-top:22px'>Processing Pipeline</div>
              <div class='ts-steps'>
              1. Map 23 attack signatures → 5 classes <code>normal · dos · probe · r2l · u2r</code><br>
              2. Encode <code>protocol_type</code> and <code>flag</code>; drop <code>service</code>, constant and highly correlated columns → <b>30 features</b><br>
              3. 67 / 33 train–test split (<code>random_state=42</code>) and MinMax scaling<br>
              4. Train Naive Bayes, Decision Tree, Random Forest, SVM, Logistic Regression, Gradient Boosting<br>
              5. Evaluate accuracy, macro precision / recall / F1 and confusion matrices</div>""")

    gr.HTML("<div class='ts-foot'>ThreatShield AI · Built on the KDD Cup 1999 dataset · For research and educational use</div>")

    # ── Wiring ──
    load_btn.click(load_data, [src, up], [kpis, p1, p2, p3, p4, preview, status])
    train_btn.click(train_models, [model_sel, fast],
                    [table, acc_plot, time_plot, detail_model, cm_plot, imp_plot, an_model, sc_model, banner, status])
    detail_model.change(show_model_details, detail_model, [cm_plot, imp_plot])
    sample_btn.click(load_sample, sample_kind, ordered)
    analyze_btn.click(analyze_record, [an_model] + ordered, [verdict, proba])
    sim_btn.click(simulate_traffic, [sc_model, n_sim], [scan_kpi, scan_plot, scan_table, scan_file_out])
    scan_btn.click(scan_file, [sc_model, csv_in], [scan_kpi, scan_plot, scan_table, scan_file_out])


## 7 · 🚀 Launch ThreatShield AI

In [ ]:
# 🚀 Launch ThreatShield AI — click the public link (or use the inline view below)
demo.queue().launch(share=True, debug=False, show_error=True)
